# Race Discovery Ingestion Pipeline
Preps race discovery CSV data for upload into `silver_race` and `silver_race_event` tables in Access.

**Workflow:**
1. Configure defaults and mappings
2. Load CSV
3. Generate IDs
4. Preview staged records
5. Write to Access database

In [1]:
# ── Cell 1: Imports ──────────────────────────────────────────────────────────
import pandas as pd
import re
from pathlib import Path
from IPython.display import display, HTML

# Access DB connection (requires 32-bit Python or Microsoft Access Database Engine)
# Install: pip install pyodbc
try:
    import pyodbc
    PYODBC_AVAILABLE = True
except ImportError:
    PYODBC_AVAILABLE = False
    print("⚠️  pyodbc not installed — preview will work but DB write will be skipped.")
    print("   Run: pip install pyodbc")

In [14]:
# ── Cell 2: Configuration ─────────────────────────────────────────────────────
# ▶ Set your CSV path
CSV_PATH = "outputs/race_discovery/race_discovery_20260511.csv"

# ▶ Path to your Access .accdb file
ACCESS_DB_PATH = r"C:\Users\Ben Sylve\Documents\git_sf\sf-pipeline\db\sf_race_metadata.accdb"

# ── Default FK values for fields not in the CSV ───────────────────────────────
# Leave as None if you want them stored as NULL
DEFAULT_RACE_LOCATION_ID   = "MS_JXN_WC"   # e.g. "LOC_JXN_MS"
DEFAULT_RACE_COURSE_ID     = "MS_JXN_RC1"   # e.g. "CRS_WC_5K"
DEFAULT_DATA_SOURCE_ID     = "RUNSIGNUP_V1"  # e.g. "RSU" for RunSignUp

# ── Distance mapping ──────────────────────────────────────────────────────────
# Maps raw CSV distance strings → your silver_race_distance keys
DISTANCE_MAP = {
    "5K"        : "5K",
    "10K"       : "10K",
    "HALF"      : "HM",
    "HALF MARATHON": "HM",
    "MARATHON"  : "FM",
    "26.2"      : "FM",
    "13.1"      : "HM",
}

# ── ID generation config ──────────────────────────────────────────────────────
# race_id format  : {ABBREV}_{CITY_CODE}_{STATE}   e.g. "WC_JXN_MS"
# event_id format : {ABBREV}_{YEAR}                e.g. "WC_2025"
#
# The abbreviation is built automatically from the race name; override here
# if the auto-generated one conflicts with an existing ID.
RACE_ABBREV_OVERRIDE = {}  # { source_race_id: "MY_ABBREV" }  e.g. {32802: "WMC"}

print("✅ Configuration loaded.")

✅ Configuration loaded.


In [3]:
# ── Cell 3: ID Generation Helpers ─────────────────────────────────────────────

def make_race_abbrev(race_name: str, max_len: int = 4) -> str:
    """
    Derives a short uppercase abbreviation from a race name.
    Strategy: initials of significant words (skips stopwords).
    e.g. 'Farm Bureau Watermelon Classic 5k Run/Walk' → 'WC'
    """
    stopwords = {"the", "a", "an", "and", "or", "of", "at", "in",
                 "run", "walk", "run/walk", "race"}
    # Strip distance tokens
    name_clean = re.sub(r'\b(5k|10k|half|marathon|\d+\.\d+)\b', '', race_name, flags=re.IGNORECASE)
    words = [w for w in name_clean.split() if w.lower() not in stopwords and len(w) > 1]
    abbrev = "".join(w[0].upper() for w in words)[:max_len]
    return abbrev if abbrev else "XX"


def make_city_code(city: str) -> str:
    """3-letter city code from city name."""
    city_clean = re.sub(r'[^A-Za-z ]', '', city).split()
    if len(city_clean) == 1:
        return city_clean[0][:3].upper()
    # Multi-word city → initials
    return "".join(w[0].upper() for w in city_clean)[:3]


def make_race_id(race_name: str, city: str, state: str, src_race_id, override_map: dict) -> str:
    abbrev = override_map.get(src_race_id) or make_race_abbrev(race_name)
    city_code = make_city_code(city)
    return f"{abbrev}_{city_code}_{state.upper()}"


def make_event_id(race_id: str, race_date) -> str:
    abbrev = race_id.split('_')[0]          # e.g. "WC" from "WC_JXN_MS"
    year = pd.to_datetime(race_date).year
    return f"{abbrev}_{year}"


def map_distance(raw: str, distance_map: dict) -> str:
    """Map raw CSV distance string to canonical distance ID."""
    key = str(raw).strip().upper()
    return distance_map.get(key, key)  # Falls back to raw value if no match


print("✅ ID helper functions defined.")

✅ ID helper functions defined.


In [30]:
# ── Cell 4: Load & Transform CSV ──────────────────────────────────────────────
df_raw = pd.read_csv(CSV_PATH)

# Optional: filter out rows without a result set (if you only want events with results)
df_raw = df_raw[df_raw["result_set_id"].notnull()]

# Only years post 2020
df_raw = df_raw[pd.to_datetime(df_raw["start_time"]).dt.year > 2020]

print(f"Loaded {len(df_raw)} rows from {CSV_PATH}")
print(f"Columns: {list(df_raw.columns)}\n")
display(df_raw)

Loaded 5 rows from outputs/race_discovery/race_discovery_20260511.csv
Columns: ['race_id', 'race_name', 'race_description', 'event_id', 'event_name', 'distance', 'start_time', 'city', 'state', 'zipcode', 'race_url', 'result_set_id', 'result_set_name']



,race_id,race_name,race_description,event_id,event_name,distance,start_time,city,state,zipcode,race_url,result_set_id,result_set_name
1,32802,Farm Bureau Watermelon Classic 5k Run/Walk,The Annual Farm Bureau Watermelon Classic bene...,987494,2025 Farm Bureau Watermelon Classic 5k RUN,5K,7/4/2025 07:30,Jackson,MS,39216,https://runsignup.com/Race/MS/Jackson/Watermel...,564071.0,Overall Results - 2025 Farm Bureau Watermelon ...
2,32802,Farm Bureau Watermelon Classic 5k Run/Walk,The Annual Farm Bureau Watermelon Classic bene...,843361,2024 Farm Bureau Watermelon Classic 5k RUN,5K,7/4/2024 07:30,Jackson,MS,39216,https://runsignup.com/Race/MS/Jackson/Watermel...,470662.0,5K Run Overall Results
3,32802,Farm Bureau Watermelon Classic 5k Run/Walk,The Annual Farm Bureau Watermelon Classic bene...,698223,2023 Farm Bureau Watermelon Classic 5k RUN,5K,7/4/2023 07:30,Jackson,MS,39216,https://runsignup.com/Race/MS/Jackson/Watermel...,391755.0,Overall Results
4,32802,Farm Bureau Watermelon Classic 5k Run/Walk,The Annual Farm Bureau Watermelon Classic bene...,609109,2022 Farm Bureau Watermelon Classic 5k RUN,5K,7/4/2022 07:30,Jackson,MS,39216,https://runsignup.com/Race/MS/Jackson/Watermel...,326598.0,Farm Bureau Watermelon Classic 5k RUN Results
5,32802,Farm Bureau Watermelon Classic 5k Run/Walk,The Annual Farm Bureau Watermelon Classic bene...,495705,2021 Farm Bureau Watermelon Classic 5k RUN,5K,7/3/2021 07:30,Jackson,MS,39216,https://runsignup.com/Race/MS/Jackson/Watermel...,261357.0,2021 Farm Bureau Watermelon Classic 5K Run Ove...


In [31]:
# ── Cell 5: Build silver_race records ─────────────────────────────────────────
# One row per unique source race_id
df_races_raw = df_raw.drop_duplicates(subset=["race_id"]).copy()

df_race = pd.DataFrame()
df_race["race_id"] = df_races_raw.apply(
    lambda r: make_race_id(r["race_name"], r["city"], r["state"],
                            r["race_id"], RACE_ABBREV_OVERRIDE), axis=1
)
df_race["race_name"]        = df_races_raw["race_name"].values
df_race["race_description"] = df_races_raw["race_description"].values

print(f"🗂  silver_race: {len(df_race)} record(s) staged")
display(df_race)


🗂  silver_race: 1 record(s) staged


,race_id,race_name,race_description
1,FBWC_JAC_MS,Farm Bureau Watermelon Classic 5k Run/Walk,The Annual Farm Bureau Watermelon Classic bene...


In [32]:
# ── Cell 6: Build silver_race_event records ───────────────────────────────────
# Build a lookup: source race_id → generated race_id
race_id_lookup = dict(zip(
    df_races_raw["race_id"],
    df_race["race_id"]
))

df_event = pd.DataFrame()

# Generated race_id (FK)
df_event["race_id"] = df_raw["race_id"].map(race_id_lookup)

# Generated race_event_id
df_event["race_event_id"] = df_raw.apply(
    lambda r: make_event_id(race_id_lookup[r["race_id"]], r["start_time"]), axis=1
)

df_event["race_event_name"]  = df_raw["event_name"]
df_event["race_date"]        = pd.to_datetime(df_raw["start_time"]).dt.date
df_event["race_start_hour"]  = pd.to_datetime(df_raw["start_time"]).dt.strftime("%H%M")

# Metadata slots — source system IDs go here
df_event["race_event_metadata_1_key"]   = "RACE_ID"
df_event["race_event_metadata_1_value"] = df_raw["race_id"].astype(str)
df_event["race_event_metadata_2_key"]   = "EVENT_ID"
df_event["race_event_metadata_2_value"] = df_raw["event_id"].astype(str)
df_event["race_event_metadata_3_key"]   = "RESULT_SET"
df_event["race_event_metadata_3_value"] = df_raw["result_set_id"].astype(str).replace("nan", None)

# FK defaults
df_event["race_distance_id"]        = df_raw["distance"].apply(lambda x: map_distance(x, DISTANCE_MAP))
df_event["race_location_id"]        = DEFAULT_RACE_LOCATION_ID
df_event["race_course_id"]          = DEFAULT_RACE_COURSE_ID
df_event["race_event_data_source_id"] = DEFAULT_DATA_SOURCE_ID

print(f"🗂  silver_race_event: {len(df_event)} record(s) staged")
display(df_event)


🗂  silver_race_event: 5 record(s) staged


,race_id,race_event_id,race_event_name,race_date,race_start_hour,race_event_metadata_1_key,race_event_metadata_1_value,race_event_metadata_2_key,race_event_metadata_2_value,race_event_metadata_3_key,race_event_metadata_3_value,race_distance_id,race_location_id,race_course_id,race_event_data_source_id
1,FBWC_JAC_MS,FBWC_2025,2025 Farm Bureau Watermelon Classic 5k RUN,2025-07-04,0730,RACE_ID,32802,EVENT_ID,987494,RESULT_SET,564071.0,5K,MS_JXN_WC,MS_JXN_RC1,RUNSIGNUP_V1
2,FBWC_JAC_MS,FBWC_2024,2024 Farm Bureau Watermelon Classic 5k RUN,2024-07-04,0730,RACE_ID,32802,EVENT_ID,843361,RESULT_SET,470662.0,5K,MS_JXN_WC,MS_JXN_RC1,RUNSIGNUP_V1
3,FBWC_JAC_MS,FBWC_2023,2023 Farm Bureau Watermelon Classic 5k RUN,2023-07-04,0730,RACE_ID,32802,EVENT_ID,698223,RESULT_SET,391755.0,5K,MS_JXN_WC,MS_JXN_RC1,RUNSIGNUP_V1
4,FBWC_JAC_MS,FBWC_2022,2022 Farm Bureau Watermelon Classic 5k RUN,2022-07-04,0730,RACE_ID,32802,EVENT_ID,609109,RESULT_SET,326598.0,5K,MS_JXN_WC,MS_JXN_RC1,RUNSIGNUP_V1
5,FBWC_JAC_MS,FBWC_2021,2021 Farm Bureau Watermelon Classic 5k RUN,2021-07-03,0730,RACE_ID,32802,EVENT_ID,495705,RESULT_SET,261357.0,5K,MS_JXN_WC,MS_JXN_RC1,RUNSIGNUP_V1


In [33]:
# ── Cell 7: ID Conflict Check ─────────────────────────────────────────────────
# Flag duplicate event IDs (can happen if a race runs twice in one year)
dup_event_ids = df_event[df_event.duplicated(subset=["race_event_id"], keep=False)]

if not dup_event_ids.empty:
    print("⚠️  Duplicate race_event_id values detected — review before writing:")
    display(dup_event_ids[["race_event_id", "race_event_name", "race_date"]])
    print()
    print("💡 Tip: For races with multiple events in the same year, append a suffix.")
    print("   Edit the make_event_id() helper to add a sequence (e.g. WC_2025_A, WC_2025_B)")
    print("   or set RACE_ABBREV_OVERRIDE to differentiate the race abbreviation.")
else:
    print("✅ No duplicate race_event_id values found.")

dup_race_ids = df_race[df_race.duplicated(subset=["race_id"], keep=False)]
if not dup_race_ids.empty:
    print("\n⚠️  Duplicate race_id values detected:")
    display(dup_race_ids)
else:
    print("✅ No duplicate race_id values found.")


✅ No duplicate race_event_id values found.
✅ No duplicate race_id values found.


In [34]:
# ── Cell 8: Preview ───────────────────────────────────────────────────────────
print("═" * 70)
print("  PREVIEW — silver_race")
print("═" * 70)
display(df_race.style.set_properties(**{'text-align': 'left'}))

print()
print("═" * 70)
print("  PREVIEW — silver_race_event")
print("═" * 70)

# Transpose for readability when there are many columns
display(
    df_event.style
        .set_properties(**{'text-align': 'left'})
        .set_table_styles([{'selector': 'th', 'props': [('text-align', 'left')]}])
)

print()
print("📋 Review the records above, then run Cell 9 to write to the database.")
print("   Set CONFIRMED = True in Cell 9 when you're ready.")


══════════════════════════════════════════════════════════════════════
  PREVIEW — silver_race
══════════════════════════════════════════════════════════════════════


,race_id,race_name,race_description
1,FBWC_JAC_MS,Farm Bureau Watermelon Classic 5k Run/Walk,"The Annual Farm Bureau Watermelon Classic benefitting the Mississippi Sports Hall of Fame & Museum will take place on Saturday, July 4, 2026! This annual event is one of the largest fundraisers for th"



══════════════════════════════════════════════════════════════════════
  PREVIEW — silver_race_event
══════════════════════════════════════════════════════════════════════


,race_id,race_event_id,race_event_name,race_date,race_start_hour,race_event_metadata_1_key,race_event_metadata_1_value,race_event_metadata_2_key,race_event_metadata_2_value,race_event_metadata_3_key,race_event_metadata_3_value,race_distance_id,race_location_id,race_course_id,race_event_data_source_id
1,FBWC_JAC_MS,FBWC_2025,2025 Farm Bureau Watermelon Classic 5k RUN,2025-07-04,0730,RACE_ID,32802,EVENT_ID,987494,RESULT_SET,564071.0,5K,MS_JXN_WC,MS_JXN_RC1,RUNSIGNUP_V1
2,FBWC_JAC_MS,FBWC_2024,2024 Farm Bureau Watermelon Classic 5k RUN,2024-07-04,0730,RACE_ID,32802,EVENT_ID,843361,RESULT_SET,470662.0,5K,MS_JXN_WC,MS_JXN_RC1,RUNSIGNUP_V1
3,FBWC_JAC_MS,FBWC_2023,2023 Farm Bureau Watermelon Classic 5k RUN,2023-07-04,0730,RACE_ID,32802,EVENT_ID,698223,RESULT_SET,391755.0,5K,MS_JXN_WC,MS_JXN_RC1,RUNSIGNUP_V1
4,FBWC_JAC_MS,FBWC_2022,2022 Farm Bureau Watermelon Classic 5k RUN,2022-07-04,0730,RACE_ID,32802,EVENT_ID,609109,RESULT_SET,326598.0,5K,MS_JXN_WC,MS_JXN_RC1,RUNSIGNUP_V1
5,FBWC_JAC_MS,FBWC_2021,2021 Farm Bureau Watermelon Classic 5k RUN,2021-07-03,0730,RACE_ID,32802,EVENT_ID,495705,RESULT_SET,261357.0,5K,MS_JXN_WC,MS_JXN_RC1,RUNSIGNUP_V1



📋 Review the records above, then run Cell 9 to write to the database.
   Set CONFIRMED = True in Cell 9 when you're ready.


In [36]:
# ── Cell 9: Write to Access Database ──────────────────────────────────────────
# ▶ Set to True only after reviewing the preview above
CONFIRMED = True

if not CONFIRMED:
    print("⏸  Write skipped — set CONFIRMED = True to proceed.")
elif not PYODBC_AVAILABLE:
    print("❌ pyodbc is not installed. Run: pip install pyodbc")
else:
    conn_str = (
        r"DRIVER={Microsoft Access Driver (*.mdb, *.accdb)};"
        f"DBQ={ACCESS_DB_PATH};"
    )
    conn = pyodbc.connect(conn_str)
    cursor = conn.cursor()

    # ── Insert silver_race ────────────────────────────────────────────────────
    race_insert_sql = """
        INSERT INTO silver_race (race_id, race_name, race_description)
        VALUES (?, ?, ?)
    """
    race_inserted = 0
    race_skipped  = 0

    for _, row in df_race.iterrows():
        # Check for existing race_id before inserting
        cursor.execute("SELECT COUNT(*) FROM silver_race WHERE race_id = ?", row["race_id"])
        exists = cursor.fetchone()[0]
        if exists:
            print(f"  ⏭  Skipping race_id '{row['race_id']}' — already exists.")
            race_skipped += 1
        else:
            cursor.execute(race_insert_sql, (
                row["race_id"],
                row["race_name"],
                row["race_description"]
            ))
            race_inserted += 1

    # ── Insert silver_race_event ──────────────────────────────────────────────
    event_insert_sql = """
        INSERT INTO silver_race_event (
            race_event_id, race_event_name, race_date, race_start_hour,
            race_event_metadata_1_key, race_event_metadata_1_value,
            race_event_metadata_2_key, race_event_metadata_2_value,
            race_event_metadata_3_key, race_event_metadata_3_value,
            race_id, race_distance_id, race_location_id,
            race_course_id, race_event_data_source_id
        ) VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
    """
    event_inserted = 0
    event_skipped  = 0

    for _, row in df_event.iterrows():
        cursor.execute(
            "SELECT COUNT(*) FROM silver_race_event WHERE race_event_id = ?",
            row["race_event_id"]
        )
        exists = cursor.fetchone()[0]
        if exists:
            print(f"  ⏭  Skipping race_event_id '{row['race_event_id']}' — already exists.")
            event_skipped += 1
        else:
            cursor.execute(event_insert_sql, (
                row["race_event_id"],
                row["race_event_name"],
                str(row["race_date"]),
                row["race_start_hour"],
                row["race_event_metadata_1_key"],
                row["race_event_metadata_1_value"],
                row["race_event_metadata_2_key"],
                row["race_event_metadata_2_value"],
                row["race_event_metadata_3_key"],
                row["race_event_metadata_3_value"],
                row["race_id"],
                row["race_distance_id"],
                row["race_location_id"],
                row["race_course_id"],
                row["race_event_data_source_id"]
            ))
            event_inserted += 1

    conn.commit()
    conn.close()

    print()
    print("═" * 50)
    print(f"  silver_race       → {race_inserted} inserted, {race_skipped} skipped")
    print(f"  silver_race_event → {event_inserted} inserted, {event_skipped} skipped")
    print("═" * 50)
    print("✅ Database write complete.")



══════════════════════════════════════════════════
  silver_race       → 1 inserted, 0 skipped
  silver_race_event → 5 inserted, 0 skipped
══════════════════════════════════════════════════
✅ Database write complete.
